# 9. T5 Relative Position Bias：从相对位置向量到距离偏置

## 一、前面的绝对位置与相对位置，解决了什么问题？

原始 Transformer 的 **Sinusoidal / Learned Absolute PE** 为每个 Token 提供位置 $p_i$，加到内容表示中，再计算 $Q,K,V$：

$$h_i=x_i+p_i,\qquad q_i=h_iW_Q,\qquad k_j=h_jW_K.$$

它并非不能感知前后方向：两个绝对位置向量参与 Attention 后，模型可以学习位置关系；只是**没有把 $j-i$ 显式交给 Attention**。

Shaw 进一步利用 **Query–Key 的相对距离**：

$$
D_{ij}=j-i,\qquad
s_{ij}^{\text{Shaw}}=\frac{q_i^\top k_j+q_i^\top a^K_{ij}}{\sqrt{d_k}}.
$$

$a^K_{ij}$ 是距离 $D_{ij}$ 对应的**可学习位置向量**。同一距离对不同 Query 可以有不同的位置贡献，因为 $q_i$ 不同。Shaw 还可以在输出侧加入相对 Value 向量。

Transformer-XL 依然保留了 **Query 与位置向量的交互**，但改为从相对距离生成 Sinusoidal 向量，经 $W_R$ 投影后参与四项 Attention Score；其跨段长上下文能力还依赖**另一个机制——历史记忆**，不是单靠位置编码。

接下来考虑一个不同的设计需求：如果位置项仅需表示**某种相对距离值得额外加多少分**，不要求这个附加分数再与 Query 内容做点积，该怎样实现？

可以直接给距离 $d$ 一个可学习标量 $b_d$，在 Softmax 之前加入内容分数：

$$
\underbrace{q_i^\top k_j}_{\text{内容匹配}}
+\underbrace{b_{j-i}}_{\text{位置偏置}}.
$$

例如，两个 Key 的内容分数都为 $0$，位置偏置分别取 $\ln 2$ 和 $0$，Softmax 后它们的权重分别为 $2/3$、$1/3$。一个标量就能改变注意力分配。

**设计取舍：** 这样减少了位置向量与 Query 的额外点积，但位置附加项不再直接依赖 $q_i$；完整 Attention 仍通过 $q_i^\top k_j$ 区分不同 Query。T5 采用的就是这种**按相对位置学习标量偏置**的方案，并为它设计了有限数量的距离类别。

## 二、为什么距离偏置还需要分桶？

最直接的实现是：距离 $0$ 查 $b_0$、距离 $1$ 查 $b_1$、距离 $2$ 查 $b_2$……每一种距离分别学习一个数。

但是序列长度并不固定。假设只考虑因果 Attention（只看当前与过去），要区分距离 $0\sim999$，每个 Head 就需要 $1000$ 个独立偏置。如果希望支持更远的距离，参数表还要扩展，并且很远的距离不一定能得到充分训练。

可以先固定偏置数量，例如**每个 Head 只分配 32 个数**。这时必须让部分距离共享同一个偏置。关键变成了：**哪些距离应该共享？**

一种简单选择是距离截断 `min(d, 31)`：距离 $0\sim30$ 分别对应不同偏置，而 $31,50,100,200$ 全部使用同一个偏置。这能控制参数数量，却使位置偏置在距离超过阈值后立刻失去进一步区分远近的能力。

如果距离 $1$ 和 $2$ 的区别值得精细区分，而距离 $100$ 和 $101$ 的区别没那么重要，那么不必给每个远距离都分配独立参数。可以让**越远的距离，共享偏置的区间越宽**，把省下的类别留给不同层次的远距离。

因此，在固定 32 个偏置的前提下，可以把距离分为两部分：

- **较近的距离：** 一个距离对应一个类别，保留具体间隔。
- **较远的距离：** 相邻的一段距离对应同一个类别；距离越远，区间逐渐变宽。

T5 使用对数增长的区间实现这一分配方式。这里的“近距离需要更细、远距离可以更粗”是**一种位置建模假设**，并不意味着远距离 Token 的内容不重要：内容是否值得关注仍由 $q_i^\top k_j$ 决定。需要精确远距离间隔的任务也可能因为合并距离而丢失信息。

### 与直接 `clamp` 的具体差别

以 **因果 Attention、32 个桶、`max_distance=128`** 为例：

| 距离 $d$ | 截断 `min(d,31)` | T5 的桶索引 |
|---:|---:|---:|
| 1 | 1 | 1 |
| 10 | 10 | 10 |
| 20 | 20 | 17 |
| 32 | 31 | 21 |
| 50 | 31 | 24 |
| 100 | 31 | 30 |
| 200 | 31 | 31 |

这些都是**查表索引**，不是偏置数值。比如 $32,50,100$ 在直接截断下已归为同一类，T5 仍能为它们分配不同的可学习偏置。代价是部分中远距离被提前合并。

**分桶不会改变标准 Attention 的 $O(L^2)$ 内容匹配开销。** 它控制的是距离偏置的参数量和距离表示的精度分配。

## 三、T5 的桶编号公式为什么这样设计？

### 1. 先分清三个量

- **距离 $d$**：Query 和 Key 相隔多少个 Token，例如 $d=20$。
- **桶号**：将距离映射到哪一个可学习偏置的索引，例如桶 $17$。
- **偏置值**：训练时该桶学出的具体标量；它**不是**距离，也**不是**桶号。

以因果 Self-Attention 为例，设置 `num_buckets=32`，桶号合法范围为 $0\sim31$。T5 将前 $16$ 个桶给短距离：$d=0,1,\ldots,15$ 分别使用 $0,1,\ldots,15$ 号桶。**其余 $16$ 个桶（编号 16～31）用来覆盖 $d\ge16$ 的更大范围。**

问题是：如果距离可以是 16、20、50、128，甚至 4096，怎样把它们分配到仅有的 16 个位置上？

### 2. 为什么远距离采用对数刻度？

假设我们希望：距离越远，一个桶能覆盖越宽的距离区间。换句话说，距离**每翻一倍**，占用的桶数大致相同。比如 $16\to32$、$32\to64$、$64\to128$ 都是翻倍，虽然相差的绝对距离分别是 16、32、64。

对数恰好具有这一性质：

$$\log(2d)-\log(d)=\log2$$

所以先决定用 **对数刻度**来压缩远距离，再考虑如何让结果恰好对应剩余的 16 个桶。这里的对数是一种设计选择，并非数学上唯一的分桶方案。

### 3. 从一个距离 $d$ 推到桶号

设精确区间结束于 $16$，并选择 `max_distance=128` 作为远距离映射的**饱和尺度**。它不是实际输入 $d$ 的最大可能值，而是我们希望到达最后一个桶时所采用的距离尺度。

**第一步：$d/16$，统一以距离 16 为起点。**

$$\frac{d}{16}$$

例如：$d=16\to1$，$d=32\to2$，$d=128\to8$。

**注意：这一步还不能确定桶编号。** $d/16$ 只是“当前距离是起点的多少倍”，既没有把结果限制到 $0\sim1$，也没有映射到后 16 个桶。

**第二步：取对数，让距离翻倍对应相同的刻度增量。**

$$f(d)=\log(d/16)$$

此时 $d=16\to0$，$d=128\to\log8$。刻度虽然从 0 开始，终点却不是 1。

**第三步：归一化，让选定范围 16～128 在对数刻度上对应 0～1。**

$$t(d)=\frac{\log(d/16)}{\log(128/16)}$$

这里的分母是**整个目标区间在同一对数刻度上的最大刻度**：

$$t(16)=0,\qquad t(128)=\frac{\log8}{\log8}=1$$

因此，分母不是在防止“桶号超过 16”，而是在设定**哪段实际距离对应后半部分的完整 0～1 比例**。分子、分母都以起点 16 为基准，是为了让端点正确对齐。

**第四步：比例乘以 16，再加 16，映射到剩余桶的编号范围。**

$$16+16\,t(d)$$

- 乘 $16$：把 $0\sim1$ 的比例扩展到剩余的 **16 个桶位**；
- 加 $16$：因为这组桶的编号从 **16** 开始，而不是从 0 开始。

**第五步：取整数，限制合法桶号。**

$$\boxed{\operatorname{bucket}(d)=\min\!\left(31,\;16+\left\lfloor16\frac{\log(d/16)}{\log(128/16)}\right\rfloor\right)},\quad d\ge16$$

对 $0\le d<16$，仍直接使用 $\operatorname{bucket}(d)=d$。当 $d=128$ 时，未截断的结果是 32；因为桶号只允许 0～31，最终为 31。更大的距离也保持为 31。

### 4. 用具体数字检验范围分配

| 距离 $d$ | $d/16$ | 归一化后 $t(d)$ | 最终桶号 |
|---:|---:|---:|---:|
| 16 | 1 | 0 | 16 |
| 20 | 1.25 | $\log(1.25)/\log8\approx0.1073$ | 17 |
| 32 | 2 | $1/3$ | 21 |
| 64 | 4 | $2/3$ | 26 |
| 128 | 8 | 1 | 31 |

例如 $d=20$：

$$16+\left\lfloor16\times0.1073\right\rfloor=17$$

这表示**距离 20 查第 17 号桶**，并不是它的偏置值等于 17。

### 5. 如果实际距离是 4096，或把 `max_distance` 改成 4096，会怎样？

这是理解分母最有用的例子。距离 $d$ 可能远远超过 128；**`max_distance` 是人为设定的映射范围，并非程序禁止的最大输入距离。**

- 若 `max_distance=128` 不变，$d=4096$ 计算出的连续桶位置会超过范围，但最终经过 `clamp(max=31)`，仍得到桶 **31**。
- 若改成 `max_distance=4096`，分母变为 $\log(4096/16)=\log256$。现在 $d=4096$ 才到归一化终点 1；相同的 $d=128$ 只有 $\log8/\log256=3/8$，对应桶 $16+\lfloor16\times3/8\rfloor=22$。

所以，改变 `max_distance` 并不是增加桶数量，**仍然只有 32 个桶，只是把后 16 个桶分配到更大的距离范围**，因而原本的各个距离可能被分到更靠前的桶。

如果**直接去掉分母归一化**，原始计算值也可能在某些距离下大于 1，乘 16 后超出剩余桶区间；`clamp` 仍会保证最终索引合法，但会改变哪些距离过早或过晚地进入最后一个桶。因此分母主要决定**距离范围与桶编号如何对齐**，而不是单纯解决数组越界。

### 6. 和代码中的变量对上

- `n`：当前方向使用的桶数；因果情况为 32。
- `max_exact = n // 2`：精确距离对应的桶数；这里是 16。
- `distance / max_exact`：距离相对精确区间起点的倍数，即 $d/16$。
- `ratio = log(d/max_exact) / log(max_distance/max_exact)`：把选定对数距离范围归一化为比例。
- `ratio * (n-max_exact)`：把比例放大到剩余的 16 个桶位。
- `+ max_exact`：偏移到后半部分桶号（从 16 开始）。
- `clamp(max=n-1)`：超过设定范围时统一使用最后一个合法桶。
- `torch.where`：短距离直接使用 $d$，其余使用上述结果。

双向 Attention 仍采用相同的分配思想，但需先区分正负距离，并将桶数分给两个方向。因果 Mask 决定哪些 Key 能被关注，桶号决定允许关注时使用什么位置偏置。

In [5]:
# 这里就是演示一下相对距离矩阵的处理，一种是transformer-xl的clamp，一种就是T5

import torch
import torch.nn as nn
import math 

def relative_position_bucket(relative_position, bidirectional=True, num_buckets=32, max_distance=128):
    # relative_position: [Q_len, K_len] 表示相对距离矩阵，就是[:;None]-[None;:]。但是要注意，T5的源代码刚好跟前面学的反过来，是用K_len-Q_len，注意这个符号的区别
    # bidirectional: 是否是双向的，这是两个应用场景，一种是单向注意力(不能看到自己后面的token)，一种是双向注意力(可以看到自己后面的token)
    # num_buckets: 相对距离矩阵的桶的数量，手动设置的
    # max_distance: 分桶的承受的位置范围，手动设置的。例如，32个桶来表示128个位置

    # 先进行判断，因为说了有两个应用场景，也可以只写一个双向的，都差不多，就是分桶的区别
    if bidirectional:
        n = num_buckets // 2
        # 双向是为了兼顾全局，因为如果是第二个token，其实前面就用不到那么多桶，但是如果对每一个token单独设计的话，太亏了。
        # 既然是双向的，那就得知道哪个正向哪个负向，如果每一次都判断这个数是否小于0太麻烦了，核心还是张量、矩阵运算，所以干脆利用相对距离矩阵构造0，1矩阵。等于0的就是负向，等于1的就是正向
        # 例如 D=[-3,0,3]
        # D>0 -> [False,False,True];'.to(torch.long)'会把False 转换成 0，True 转换成 1
        relative_buckets = (relative_position > 0).to(torch.long) 
        # 这一步很关键，我们有num_buckets个桶，前一半是负向的，后一半是正向的，所以我们要把正向的桶偏移到后一半
        # 例如：0-15 负责负向，16-31 负责正向。为什么又要从0开始呢？因为后续的嵌入读取的索引得从0开始读，参考一下shaw位置编码 
        # 负的偏移为0，正的偏移为16
        direction_offset = relative_buckets * n

        # 因为0-15桶负责负向，我们利用log计算的时候，需要的是正数，所以需要取绝对值
        # 这里不要跟上面的direction_offset这些混淆了。这里只是把原来的相对矩阵取绝对值，便于后续判断哪个位置直接使用位置来取嵌入
        relative_position = torch.abs(relative_position)

    else:
        # 只看过去，不看未来。所以没有必要把未来的位置加上桶的数量作为偏移
        direction_offset = 0
        # 负向表示该tokne在左边，相当于看未来。正向的才是看过去
        # clamp(min=0) 表示小于 0 的数全部改为 0
        # relative_position = (-relative_position).clamp(min=0) # 源码写法，因为相对矩阵的计算方式跟shaw的反过来了，所以token在左边是负向(过去)，token在右边是正向
        # [-2，0，3] -> [0,0,3]
        n = num_buckets
        relative_position = relative_position.clamp(min=0) # 这里我改成了正向的，方便跟前面对接，但是T5的更好理解一点


    # 下一步就是在单向或者双向中继续区分，一半的桶用于近距离，一半的桶用于远距离。近距离就是直接映射到桶上，远距离就是用log映射到桶上
    # 单向 Attention：n=32，前16个桶用于近距离，后16个桶用于远距离
    # 双向 Attention：n=16，每个方向前8个桶用于近距离，后8个桶用于远距离
    max_exact = n // 2

    # 下一步就是开始分桶了：判断每个距离是否属于近距离。这里就是用上我们上一步做的relative_position。双向的话会有重复值的，但是别担心[3,2,1,0,1,2,3]
    # 小于近距离桶边界就可以直接用原来的位置了。用别的写法也行，能实现这个想法就好。
    is_small = relative_position < max_exact 

    # 判断远距离位置在的桶编号，这里就需要按照公式进行计算了。
    # 把近距离的桶先都调整为边界值，这样子使用的时候依旧可以矩阵计算，不用去遍历
    # 转换成浮点数，因为后面要进行浮点数计算，这样可以少损失精度
    safe_distance = relative_position.float().clamp(min=max_exact)

    # 开始按照公式进行计算远距离分桶
    # 分子
    relative_ratio = safe_distance / max_exact
    log_distance = torch.log(relative_ratio)
    # 分母
    log_max = math.log(max_distance / max_exact)
    ratio = log_distance / log_max
    # 远距离桶编号 = ratio * (n - max_exact)，这里的n-max_exact表示远距离桶的数量
    far_index = ratio * (n - max_exact)
    # torch.floor()：向下取整
    # .long()：转换为整数类型
    far_index = torch.floor(far_index).long()

    # far_index 目前是从0开始计算的局部编号，但近距离已经占用了前 max_exact 个桶。要加上近距离占用的桶
    far_bucket = max_exact + far_index

    # 远距离桶编号不能超过总桶数的上限，超过直接截断
    far_bucket = far_bucket.clamp(max=n - 1)

    # 最终每个相对位置对应的桶编号 = 方向偏移 + 近距离桶编号/远距离桶编号
    # torch.where(条件, True时选择的值, False时选择的值)
    # 小于近距离桶边界直接使用原来的位置，大于近距离桶边界使用远距离桶编号far_bucket
    return direction_offset + torch.where(is_small, relative_position, far_bucket)


pos = torch.arange(4)

# 按照你的定义：Query位置 - Key位置
D = pos[:, None] - pos[None, :]

print("相对距离矩阵：\n", D)

# 双向分桶
bucket = relative_position_bucket(
    D,
    bidirectional=True,
    num_buckets=32,
    max_distance=128
)

print("双向桶编号：\n", bucket)


相对距离矩阵：
 tensor([[ 0, -1, -2, -3],
        [ 1,  0, -1, -2],
        [ 2,  1,  0, -1],
        [ 3,  2,  1,  0]])
双向桶编号：
 tensor([[ 0,  1,  2,  3],
        [17,  0,  1,  2],
        [18, 17,  0,  1],
        [19, 18, 17,  0]])


## 四、为什么一个桶要对应多个 Head 的标量？

前面学过 Multi-Head Attention：不同 Head 使用不同的投影矩阵，因而能学习不同的内容匹配关系。位置偏好也可以不同。例如，相同的距离 $2$，Head 0 可能学到较高的偏置，Head 1 则可能学到较低的偏置。

因此，T5 的偏置表可以写成下面的样子，T5是偏置，所以一维就可以了，没有必要nn.Embedding(num_buckets, d_model,num_heads)：

```python
nn.Embedding(num_buckets, num_heads)
```

注意，表中**每一行不是 $d_{\mathrm{model}}$ 维的位置向量**，而是给每个 Head 各存一个标量。

假设 `bucket` 的形状为 `(L_q,L_k)`，这个形状跟相对矩阵一样的：

| 步骤 | 形状 | 原因 |
|---|---|---|
| `bucket` | `(L_q,L_k)` | 每对 Query–Key 有一个桶索引 |
| `bias_table(bucket)` | `(L_q,L_k,H)` | 每个桶查出 H 个 Head 各自的标量 |
| `permute(2,0,1)` | `(H,L_q,L_k)` | Attention Score 的 Head 维在前 |
| `unsqueeze(0)` | `(1,H,L_q,L_k)` | 与不同 Batch 共享同一套位置偏置 |
| `content_scores + bias` | `(B,H,L_q,L_k)` | Batch 维通过广播从 1 扩展到 B |

注意力分数最终是：

$$
S^{(h)}_{ij}=q_i^{(h)\top}k_j^{(h)}+b_{h,\operatorname{bucket}(j-i)}.
$$

偏置必须**加在 Softmax 前**，因为它的作用是改变各 Key 的竞争分数。T5 没有使用 Shaw 的相对 Value 项；同一 Head 中的同一桶也共享同一个位置偏置。原始 T5 实现也不额外对这里的 $QK^\top$ 除以 $\sqrt{d_k}$，与此前讲过的普通 Scaled Dot-Product Attention 写法有所不同。T5把归一化的步骤直接放在了W_Q和W_K的初始化部分了，效果是一样的，其实也无所谓，我们学的只是他对于相对位置的处理。

## 四、三种相对位置方法，分别在解决什么问题？

它们都让相对关系参与 Attention，但**位置项想表达的内容不同**。选择方法时，先看需求，再看任务名称。

| 方法 | 位置项的实际机制 | 哪种需求与它更吻合，为什么 | 代价或局限 |
|---|---|---|---|
| **Shaw RPE** | 每个距离查出可学习位置向量，利用 $q_i^\top a^K_{ij}$ 改变注意力分数；可加入相对 Value | **相同距离对不同 Query 需要有不同含义**时，可以通过 $q_i$ 与位置向量的交互区分 | 朴素实现有额外位置向量计算；超过裁剪距离后不能区分更远的精确距离 |
| **Transformer-XL Relative PE** | 相对 Sinusoidal 向量经 $W_R$ 投影，通过四项分数参与 Attention | **希望保留内容与距离的交互，并能计算跨段相对距离**时适合；长文本上下文则由其**历史记忆**提供 | 公式和实现更复杂；位置编码本身不会保存历史内容 |
| **T5 Relative Bias** | 距离分桶，每个 Head 查一个标量，直接加到 Score | **只需要学习各 Head 对距离类别的加减分偏好**时更直接；固定桶数可兼顾近距离精细区分与更远距离覆盖 | 同 Head、同桶的附加偏置相同；不直接依赖 Query 内容，远距离精度会合并 |

**应用场景由机制决定，而不是被论文名称限定。** 例如，长文本建模必须解决历史信息可访问的问题，使用 Transformer-XL 的 Hidden State 记忆或正确实现的 KV Cache 都可以；单独更换位置编码不会凭空获得历史内容。另一方面，对距离的精确依赖强弱，要结合任务与消融实验判断。

这三种机制与绝对位置编码的关系也可以统一理解：

- **绝对位置编码**提供 $p_i$，告诉模型每个 Token 的位置；模型可以从中学习前后关系，但相对距离不是显式输入。
- **相对位置编码**直接使用 $D_{ij}=j-i$（或相反的符号约定），告诉 Attention 两个 Token 相隔多远、谁在前谁在后。
- **因果 / 双向 Attention** 由 Mask 决定能看哪些 Token；位置编码使用哪种形式，不直接决定能否看未来。

### 论文结论与设计推论的边界

Shaw 与 Transformer-XL 分别提出、验证了自己的设计，但**不能把不同论文的最终任务指标直接当作三种位置编码的公平排名**。T5 的主要贡献是 Text-to-Text 迁移学习的统一研究；论文将标量相对位置偏置作为模型架构选择，并未单独用完整消融证明它在所有条件下优于 Shaw 或 Transformer-XL。

**参考资料：**

1. [Shaw et al., *Self-Attention with Relative Position Representations* (NAACL 2018)](https://aclanthology.org/N18-2074/)
2. [Dai et al., *Transformer-XL: Attentive Language Models beyond a Fixed-Length Context* (ACL 2019)](https://aclanthology.org/P19-1285/)
3. [Raffel et al., *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer* (JMLR 2020)](https://www.jmlr.org/papers/v21/20-074.html)
4. [Hugging Face Transformers：T5 相对位置分桶与 Attention 实现](https://github.com/huggingface/transformers/blob/main/src/transformers/models/t5/modeling_t5.py)